# Control 3: does routing respond to perturbation state within a cell type?

Controls 1 and 2 ask whether the CGP routing weight is a myeloid indicator.
Here we ask a different, functional question that those controls cannot see:
**is the routing weight a static cell-type label, or does it track cell
state?**

The ifnb dataset contains IFN-beta-stimulated and matched control cells for
every cell type. If the typed experts use their knowledge, stimulation should
change *how cells are routed*: cells under perturbation should be routed more
towards the perturbation-knowledge expert (CGP) and less towards the
housekeeping-pathway expert (Reactome), and this should happen inside cell
types, not merely between them.

We compute, for every expert gate:

- the partial correlation of the gate weight with the stimulation indicator,
 controlling for cell type (`r_stim | cell type`);
- the stimulation-induced weight change (stim mean minus ctrl mean) separately
 in myeloid and lymphoid cells.

We run this on the three seeds of the real model and on all six
random-grouping models. Run from the `ICLR_review/` directory (kernel: the
`deepfusion` conda environment).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import anndata

ROOT = Path("..")
IFNB = ROOT / "ifnb"
EXP05 = Path("exp05_random_grouping")
OUT = Path("exp08_e7cp_controls")
OUT.mkdir(exist_ok=True)

MYELOID = ["DC", "CD14 Mono", "CD16 Mono"]
LYMPHOID = ["CD4 Naive T", "CD4 Memory T", "CD8 T", "T activated",
            "B", "B Activated", "NK"]

ad = anndata.read_h5ad(IFNB / "enhanced_results/results/step5_add_DeepFusion.h5ad")
labels = ad.obs["seurat_annotations"].astype(str).values
s_bin = (ad.obs["stim"].astype(str).values == "STIM").astype(float)
m_myel = np.isin(labels, MYELOID)
m_lymph = np.isin(labels, LYMPHOID)
print(pd.Series(ad.obs["stim"].astype(str)).value_counts().to_dict())

ctypes = sorted(set(labels.tolist()))
Ct = [(labels == ct).astype(float) for ct in ctypes]

def partial_corr(x, y, controls):
    C = np.column_stack([np.ones(len(x))] + list(controls))
    rx = x - C @ np.linalg.lstsq(C, x, rcond=None)[0]
    ry = y - C @ np.linalg.lstsq(C, y, rcond=None)[0]
    return float(np.corrcoef(rx, ry)[0, 1])

def stim_delta(w, mask):
    return float(w[mask & (s_bin == 1)].mean() - w[mask & (s_bin == 0)].mean())

{'STIM': 7234, 'CTRL': 6420}


## Real model: routing redistribution under stimulation (three seeds)

Partial correlation of each expert's weight with the stimulation indicator,
controlling for cell type, and the stim-minus-ctrl weight change by lineage.

In [2]:
GATES = ["CGP", "Reactome", "Gene-line"]
rows = []
for s, k in [("seed42", "X_expert_weights_40_std"),
             ("seed43", "X_expert_weights_40_std_s43"),
             ("seed44", "X_expert_weights_40_std_s44")]:
    W = np.asarray(ad.obsm[k])
    for j, g in enumerate(GATES):
        rows.append({
            "seed": s, "gate": g,
            "r_stim_given_celltype": partial_corr(W[:, j], s_bin, Ct),
            "stim_minus_ctrl_myeloid": stim_delta(W[:, j], m_myel),
            "stim_minus_ctrl_lymphoid": stim_delta(W[:, j], m_lymph),
        })
real_stim = pd.DataFrame(rows)
real_stim.to_csv(OUT / "control3_real_stim_response.csv", index=False)
print(real_stim.round(3).to_string(index=False))

  seed      gate  r_stim_given_celltype  stim_minus_ctrl_myeloid  stim_minus_ctrl_lymphoid
seed42       CGP                  0.282                    0.148                     0.000
seed42  Reactome                 -0.338                   -0.351                    -0.003
seed42 Gene-line                  0.341                    0.203                     0.003
seed43       CGP                  0.142                    0.064                     0.002
seed43  Reactome                 -0.135                   -0.114                     0.004
seed43 Gene-line                  0.085                    0.050                    -0.007
seed44       CGP                  0.217                    0.089                    -0.002
seed44  Reactome                 -0.245                   -0.190                     0.001
seed44 Gene-line                  0.225                    0.101                     0.001


## Per-cell-type view (seed 42, CGP weight)

The CGP response is concentrated in the myeloid compartment; lymphoid cell
types barely move.

In [3]:
W42 = np.asarray(ad.obsm["X_expert_weights_40_std"])[:, 0]
per_ct = pd.DataFrame({
    "cell_type": ctypes,
    "w_ctrl": [W42[(labels == ct) & (s_bin == 0)].mean() for ct in ctypes],
    "w_stim": [W42[(labels == ct) & (s_bin == 1)].mean() for ct in ctypes],
})
per_ct["delta"] = per_ct.w_stim - per_ct.w_ctrl
per_ct["lineage"] = np.where(np.isin(per_ct.cell_type, MYELOID), "myeloid",
                     np.where(np.isin(per_ct.cell_type, LYMPHOID), "lymphoid", "other"))
per_ct.to_csv(OUT / "control3_per_celltype_stim_delta.csv", index=False)
print(per_ct.round(3).sort_values("delta", ascending=False).to_string(index=False))

   cell_type  w_ctrl  w_stim  delta  lineage
   CD14 Mono   0.381   0.541  0.160  myeloid
   CD16 Mono   0.385   0.536  0.151  myeloid
       Eryth   0.083   0.144  0.060    other
          DC   0.517   0.561  0.044  myeloid
         pDC   0.338   0.378  0.040    other
          NK   0.081   0.091  0.010 lymphoid
           B   0.057   0.064  0.007 lymphoid
       CD8 T   0.063   0.066  0.003 lymphoid
CD4 Memory T   0.033   0.035  0.002 lymphoid
 B Activated   0.149   0.150  0.001 lymphoid
 CD4 Naive T   0.033   0.033  0.000 lymphoid
 T activated   0.083   0.076 -0.007 lymphoid
          Mk   0.182   0.156 -0.026    other


## Random-grouping models: no consistent direction

Same analysis on the six random-grouping runs. `*` marks the gate that
prefers myeloid cells (the gate used in Control 2). Whether any gate of a
random model responds to stimulation is a coin flip, and the sign is
inconsistent — unlike the real model, where the CGP gate responds positively
in 3/3 seeds with myeloid specificity.

In [4]:
rows = []
for rep in (1, 2, 3):
    for seed in (42, 43):
        run = f"rep{rep}_seed{seed}"
        z = np.load(EXP05 / "runs" / run / "embeddings.npz", allow_pickle=True)
        w = z["X_expert_weights"]
        assert (z["label"].astype(str) == labels).all()
        dA = w[m_myel, 0].mean() - w[m_lymph, 0].mean()
        dB = w[m_myel, 1].mean() - w[m_lymph, 1].mean()
        myeloid_gate = 0 if dA >= dB else 1
        for j, g in enumerate(["RAND_A", "RAND_B", "gene"]):
            rows.append({
                "run": run, "gate": g,
                "is_myeloid_preferring_gate": j == myeloid_gate,
                "r_stim_given_celltype": partial_corr(w[:, j], s_bin, Ct),
                "stim_minus_ctrl_myeloid": stim_delta(w[:, j], m_myel),
                "stim_minus_ctrl_lymphoid": stim_delta(w[:, j], m_lymph),
            })
rand_stim = pd.DataFrame(rows)
rand_stim.to_csv(OUT / "control3_random_stim_response.csv", index=False)
print(rand_stim.round(3).to_string(index=False))

        run   gate  is_myeloid_preferring_gate  r_stim_given_celltype  stim_minus_ctrl_myeloid  stim_minus_ctrl_lymphoid
rep1_seed42 RAND_A                        True                  0.175                    0.107                     0.001
rep1_seed42 RAND_B                       False                 -0.147                   -0.114                    -0.001
rep1_seed42   gene                       False                  0.022                    0.007                    -0.001
rep1_seed43 RAND_A                       False                 -0.054                   -0.039                    -0.001
rep1_seed43 RAND_B                        True                  0.142                    0.058                     0.011
rep1_seed43   gene                       False                 -0.048                   -0.019                    -0.010
rep2_seed42 RAND_A                       False                  0.179                    0.060                     0.024
rep2_seed42 RAND_B              

## Summary

- Real model: CGP weight responds positively to stimulation in 3/3 seeds
 (partial r = 0.14-0.28), the response is specific to myeloid cells
 (lymphoid delta ~ 0), and the Reactome weight moves in the opposite
 direction in 3/3 seeds.
- Random models: the myeloid-preferring gate responds positively in only 2/6
 runs; signs are inconsistent across replicates.
- Caveat we state explicitly: the gene-level expert also responds positively
 (stimulation changes gene expression broadly), so we do not claim the
 response is exclusive to the CGP expert. We claim it is consistent,
 lineage-specific, and aligned with the CGP expert's knowledge type
 (perturbation-response signatures).

In [5]:
summ = pd.DataFrame({
    "model": ["real (3 seeds)", "random (6 runs)"],
    "myeloid-pref gate r_stim|ct > 0": ["3/3", "2/6"],
    "direction consistent": ["yes (all +, myeloid-specific)", "no (mixed signs)"],
})
print(summ.to_string(index=False))

          model myeloid-pref gate r_stim|ct > 0          direction consistent
 real (3 seeds)                             3/3 yes (all +, myeloid-specific)
random (6 runs)                             2/6              no (mixed signs)
